In [ ]:
import os
import json
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
import time
from tqdm import tqdm
import pandas as pd

CRITERIA = criteria_str = """
(1) Completeness of abnormal features mentioned (higher=more complete): 10,
(2) Completeness of key diagnoses included (higher=more complete): 10,
(3) Absence of critical diagnostic errors (higher=better): 8,
(4) Whether severity of symptoms or likelihood of condition is described: 7,
(5) Whether tentative diagnoses are included: 7,
(6) Whether the location of the issue is correctly described: 8,
(7) Whether symptoms are correctly identified: 7,
(8) Whether the basic rhythm is correctly determined: 4,
(9) Whether arrhythmia classification is correct: 4,
(10) Whether conduction abnormalities are correctly described: 4,
(11) Whether pacing signals are correctly identified: 3,
(12) Whether the report structure is clear: 5,
(13) Whether different symptoms are elaborated in separate points: 4,
(14) Whether age, gender, and other auxiliary diagnostic information are included: 3,
(15) Whether patient privacy is protected: 3,
(16) Whether terminology complies with SCP-ECG standards: 5,
(17) Whether wording is appropriate, avoiding absolute expressions: 5
"""

PRED_KEYS = ["gen_answer", "model_answer", "model_output", "claude_pred", "pred", "model_pred", "modelOutput"]

def parse_model_response(response_text: str):
    try:
        response_text = response_text.strip()
        
        if response_text.startswith("```"):
            response_text = response_text.strip("`")
            if response_text.lower().startswith("json"):
                response_text = response_text[4:].strip()
        
        parsed = json.loads(response_text)
        # print(parsed)
        return parsed
    except Exception:
        return {}

def extract_pred_from_item(item: dict) -> str:
    for k in PRED_KEYS:
        if k in item and item[k] not in (None, ""):
            return item[k]
    for k in item.keys():
        if k.lower() in PRED_KEYS:
            return item[k]
    return ""

def extract_ref_from_item(item: dict) -> str:
    ref = item.get("answer", "")
    if isinstance(ref, list):
        ref = ", ".join(ref).capitalize()
    return ref

def evaluate_report_with_gpt(gen_answer, reference_answer):
    prompt = f"""You are a professional ECG report evaluation expert. Please compare the candidate report with the reference report and score strictly according to the following criteria:

### Evaluation Criteria:
{CRITERIA}

### Requirements:
1. Score each sub-item from 0-100 based on comparison with reference 
2. Calculate weighted dimension scores (sum(sub_score × weight)/sum(weights))
3. Final total_score is the sum of all weighted dimension scores
4. You must compare reports and respond with exactly the requested JSON format. Output format must be:
{{
    "item_scores": {{
        "1": score, "2": score, ..., "17": score  
    }},
    "total_score": total_score
}}"""
    
    user = f"""### Reference Report:
{reference_answer}

### Candidate Report to Evaluate:
{gen_answer}"""

    key = ''

    try:
        client = OpenAI(
            base_url="",
            api_key=key
        )
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": user}
            ]
        )
        result = response.choices[0].message.content
        print(result)
        result = parse_model_response(result)
        
        required_keys = {"item_scores", "total_score"}
        if not all(k in result for k in required_keys):
            raise ValueError("Missing required keys in response")
            
        return result
        
    except Exception as e:
        print(f"Error evaluating report: {e}")
        return {
            "item_scores": 0,
            "total_score": 0,
        }